# 07x feature mapping AARRR 260515

06x conservative_safe_22 and expanded_feature_set 기준 feature role, AARRR mapping, scope policy, caveat handoff를 생성하는 notebook입니다.


In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import subprocess
import zipfile

import pandas as pd

STEP_NAME = '07x_feature_mapping_AARRR_260515'
ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()).resolve()
PARK = ROOT / 'park.ingyeom'
NB_DIR = PARK / 'notebook' / STEP_NAME
REPORT_DIR = PARK / 'reports' / 'audits' / STEP_NAME
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP_NAME}_review_package.zip'
SRC06 = PARK / 'reports' / 'audits' / '06x_dataset_generation_260515'
DICT05Y = PARK / 'reports' / 'audits' / '05y_feature_approval_and_dictionary_patch2_260515' / '05y_feature_dictionary.xlsx'
ARCHIVE_NB = PARK / '_archive' / 'pre13b_ref' / 'notebook' / '07_AARRR_feature_mapping_260513' / '07_AARRR_feature_mapping_260513.ipynb'
NB_PATH = NB_DIR / f'{STEP_NAME}.ipynb'
NOTE_PATH = PARK / 'note.md'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

required_06x = [
    '06x_conservative_dataset.csv',
    '06x_expanded_dataset.csv',
    '06x_model_feature_lists.csv',
    '06x_dataset_schema_conservative.csv',
    '06x_dataset_schema_expanded.csv',
    '06x_scope_feature_policy.csv',
    '06x_caveat_register.csv',
    '06x_feature_name_mapping.csv',
    '06x_final_checks.csv',
]
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
execution_events = []

def log(message):
    line = f'[{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}] {message}'
    execution_events.append(line)
    print(line)

def inside_park(path):
    p = Path(path).resolve()
    return p == PARK or PARK in p.parents

def file_fingerprint(path):
    p = Path(path)
    if not p.exists():
        return None
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    stat = p.stat()
    return {'size': stat.st_size, 'mtime_ns': stat.st_mtime_ns, 'sha256': h.hexdigest()}

raw_csv_paths = sorted((PARK / 'data').rglob('*.csv')) if (PARK / 'data').exists() else []
raw_before = {str(p.relative_to(PARK)): file_fingerprint(p) for p in raw_csv_paths}
log(f'root={ROOT}')
log(f'raw_csv_fingerprint_count={len(raw_before)}')

preflight_rows = []
def add_preflight(check, status, detail='', stop_reason=''):
    preflight_rows.append({'check': check, 'status': status, 'detail': detail, 'stop_reason': stop_reason})

add_preflight('06x_folder_exists', 'PASS' if SRC06.exists() else 'FAIL', str(SRC06), '' if SRC06.exists() else 'missing_06x_folder')
missing = [name for name in required_06x if not (SRC06 / name).exists()]
for name in required_06x:
    add_preflight(f'required_06x_file_exists::{name}', 'PASS' if (SRC06 / name).exists() else 'FAIL', str(SRC06 / name), '' if (SRC06 / name).exists() else 'missing_required_input')
add_preflight('05y_feature_dictionary_exists', 'PASS' if DICT05Y.exists() else 'FAIL', str(DICT05Y), '' if DICT05Y.exists() else 'missing_05y_dictionary')
add_preflight('archived_07_notebook_found', 'PASS' if ARCHIVE_NB.exists() else 'WARN', str(ARCHIVE_NB), '' if ARCHIVE_NB.exists() else 'archive_07_notebook_not_found_new_notebook_created')
add_preflight('output_folder_created', 'PASS' if REPORT_DIR.exists() and inside_park(REPORT_DIR) else 'FAIL', str(REPORT_DIR), '' if REPORT_DIR.exists() else 'output_folder_missing')

if missing or not SRC06.exists() or not DICT05Y.exists():
    pd.DataFrame(preflight_rows).to_csv(REPORT_DIR / '07x_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
    raise RuntimeError('Required input missing. See 07x_preflight_input_validation.csv')

final06 = pd.read_csv(SRC06 / '06x_final_checks.csv')
status_col = 'status' if 'status' in final06.columns else final06.columns[1]
final06_pass = final06[status_col].astype(str).str.upper().eq('PASS').all()
add_preflight('06x_final_checks_pass', 'PASS' if final06_pass else 'FAIL', f'{status_col} all PASS={final06_pass}', '' if final06_pass else '06x_final_checks_not_all_pass')
pd.DataFrame(preflight_rows).to_csv(REPORT_DIR / '07x_preflight_input_validation.csv', index=False, encoding='utf-8-sig')
if not final06_pass:
    raise RuntimeError('06x final checks did not all PASS')

model_features = pd.read_csv(SRC06 / '06x_model_feature_lists.csv')
schema_con = pd.read_csv(SRC06 / '06x_dataset_schema_conservative.csv')
schema_exp = pd.read_csv(SRC06 / '06x_dataset_schema_expanded.csv')
scope_policy = pd.read_csv(SRC06 / '06x_scope_feature_policy.csv')
caveat06 = pd.read_csv(SRC06 / '06x_caveat_register.csv')
name_map = pd.read_csv(SRC06 / '06x_feature_name_mapping.csv')
dict05 = pd.read_excel(DICT05Y, sheet_name='01_feature_dictionary')
formula05 = pd.read_excel(DICT05Y, sheet_name='04_formula_validation')
log(f'06x_inputs_loaded rows model_features={len(model_features)} dictionary={len(dict05)}')

schema = pd.concat([
    schema_con.assign(feature_set_name='conservative_safe_22'),
    schema_exp.assign(feature_set_name='expanded_feature_set')
], ignore_index=True)
dict_cols = ['original_feature_name','safe_model_feature_name','feature_description','feature_generation_principle','source_columns','formula','caveat_flag','caveat_description','notes']
dict_small = dict05[[c for c in dict_cols if c in dict05.columns]].copy()
merged = model_features.merge(schema[['feature_set_name','column_name','source_original_feature','dtype','missing_count','unique_count']], left_on=['feature_set_name','safe_model_feature_name'], right_on=['feature_set_name','column_name'], how='left')
merged = merged.merge(dict_small, on=['original_feature_name','safe_model_feature_name'], how='left', suffixes=('','_dict'))

caveat_lookup = {str(r['item']).lower(): str(r['caveat_reason']) for _, r in caveat06.iterrows()}
formula_lookup = {str(r['feature_name']): r.to_dict() for _, r in formula05.iterrows() if 'feature_name' in formula05.columns}

def contains_any(name, needles):
    return any(x in name for x in needles)

def classify(row):
    safe = str(row['safe_model_feature_name'])
    role0 = str(row.get('role','feature'))
    use0 = str(row.get('use_as_feature','yes')).lower() in ['yes','true','1']
    name = safe.lower()
    role = role0
    use = use0
    stage = 'Needs_user_review'
    family = 'needs_review'
    timing = 'not_time_bound'
    confidence = 'low'
    needs_review = 1
    basis = '06x_model_feature_lists + 06x_schema + name_pattern_review + 05y_dictionary'
    notes = ''
    downstream_scope = 'all_06x_scopes_subject_to_scope_policy'
    caveat_flag = bool(str(row.get('caveat_flag', False)).lower() == 'true')
    caveat_reason = '' if pd.isna(row.get('caveat_reason')) else str(row.get('caveat_reason'))

    if safe == 'USER_KEY':
        return pd.Series({'role':'group_key','use_as_feature':False,'AARRR_stage':'Identifier','feature_family':'identifier','timing_family':'not_time_bound','source_table_or_source':'membership/source master','caveat_flag':False,'caveat_reason':'USER_KEY is an identifier and group key, not a model feature.','downstream_allowed_scope':'grouping_or_join_only_not_model_feature','mapping_basis':basis,'mapping_confidence':'high','needs_user_review':0,'notes':'identifier only'})
    if safe == 'is_repurchase':
        return pd.Series({'role':'target','use_as_feature':False,'AARRR_stage':'Revenue_proxy','feature_family':'target_proxy','timing_family':'post_observation_target','source_table_or_source':'membership/source master','caveat_flag':True,'caveat_reason':'is_repurchase is the target and may be described only as a Revenue proxy, not as a feature.','downstream_allowed_scope':'target_only_not_model_feature','mapping_basis':basis,'mapping_confidence':'high','needs_user_review':0,'notes':'target/proxy only'})
    if safe == 'is_promotion':
        return pd.Series({'role':'scope_conditional_feature','use_as_feature':True,'AARRR_stage':'Acquisition','feature_family':'acquisition_split_key','timing_family':'membership_context','source_table_or_source':'membership/source master','caveat_flag':True,'caveat_reason':'Allowed as a feature only for overall_with_promotion; excluded for overall_without_promotion, promotion_only, and nonpromotion_only.','downstream_allowed_scope':'overall_with_promotion_only','mapping_basis':'explicit user rule + 06x_scope_feature_policy','mapping_confidence':'high','needs_user_review':0,'notes':'split key and scope-conditional feature'})
    if safe in ['is_cold_start_3d','is_cold_start_7d']:
        return pd.Series({'role':'audit_only_original_cold_start','use_as_feature':False,'AARRR_stage':'Activation','feature_family':'cold_start_original_excluded','timing_family':'early_observation','source_table_or_source':'source master audit only','caveat_flag':True,'caveat_reason':'Original cold_start fields are excluded; fixed replacements must be used.','downstream_allowed_scope':'not_allowed_as_model_feature','mapping_basis':'explicit user rule + 05y formula validation','mapping_confidence':'high','needs_user_review':0,'notes':'original excluded'})
    if safe in ['is_cold_start_3d_fixed','is_cold_start_7d_fixed']:
        reason = 'Fixed cold_start feature generated from row-level first watch timing; original cold_start fields are not model features.'
        return pd.Series({'role':'feature','use_as_feature':True,'AARRR_stage':'Activation','feature_family':'onboarding_cold_start_fixed','timing_family':'day0_early_observation','source_table_or_source':'View_History_v2 + User_Mapping_v2 + source master hotfix','caveat_flag':True,'caveat_reason':reason,'downstream_allowed_scope':downstream_scope,'mapping_basis':'explicit user rule + 05y formula validation + 06x hotfix','mapping_confidence':'high','needs_user_review':0,'notes':'fixed replacement used'})
    if safe == 'is_churn_prevented':
        return pd.Series({'role':'feature','use_as_feature':True,'AARRR_stage':'Retention_context','feature_family':'historical_churn_prevention_context','timing_family':'historical_context','source_table_or_source':'membership/source master','caveat_flag':True,'caveat_reason':'Historical ever-benefited churn prevention flag; do not interpret as current-cycle outcome.','downstream_allowed_scope':downstream_scope,'mapping_basis':'explicit user rule + 06x schema caveat','mapping_confidence':'high','needs_user_review':0,'notes':'expanded context/history feature'})

    if name.startswith('watch_time_min_w1') or name.startswith('watch_session_w1') or name in ['is_w1_over_50pct','is_only_w1']:
        stage, family, timing, confidence, needs_review = 'Activation', 'early_week1_watch_behavior', 'week1', 'high', 0
    elif contains_any(name, ['_w2','_w3','retention','diff_between','recency','inactive','rewatch','active_ratio','watch_days','watch_per_day','total_watch','total_watch_count','unique_movie','movie_per_active_day','weekend_watch_ratio','max_day','max_watch','max_daily_sessions','day_count_over','avg_watch','median_watch','std_watch','daily_watch','watch_ratio_under']):
        stage, family, timing, confidence, needs_review = 'Retention', 'usage_retention_behavior', 'multi_week_or_aggregate_observation', 'medium', 0
    elif contains_any(name, ['new_movie','old_movie','ott_release','genre','action_adventure','family_animation','drama','thriller','sf_fantasy','comedy','romance','horror','documentary','historical_war','other_ratio']):
        stage, family, timing, confidence, needs_review = 'Retention_context', 'content_preference_context', 'observation_window_aggregate', 'medium', 0
    elif name in ['is_standard','is_premium','is_basic','payment_is_mobile','payment_is_pc','payment_is_android','payment_is_ios','reg_is_weekend','reg_hour_morning','reg_hour_afternoon','reg_hour_evening','reg_hour_night']:
        stage, family, timing, confidence, needs_review = 'Acquisition_context', 'membership_context', 'registration_or_payment_context', 'medium', 0
        notes = 'Context feature only; not a causal acquisition measurement.'
    elif name in ['age_group','is_female','is_male','is_user_verified']:
        stage, family, timing, confidence, needs_review = 'Needs_user_review', 'profile_context_needs_review', 'membership_context', 'low', 1
        notes = 'Name alone does not prove an AARRR stage; left for user review.'

    if safe == 'old_movie_ratio_5y':
        caveat_flag = True
        caveat_reason = 'Use Kwangil master value as-is; raw Movie_Master reconstruction has 9-row mismatch caveat.'
    elif safe in ['action_adventure_ratio','family_animation_ratio','drama_ratio','thriller_crime_ratio','sf_fantasy_ratio','comedy_ratio','romance_ratio','horror_ratio','documentary_ratio','historical_war_ratio','other_ratio','genre_diversity_count']:
        caveat_flag = True
        caveat_reason = 'Movie_Master can have multiple category records for the same MOVIE_NUM; genre features require caveat handoff.'
    elif safe in ['watch_ratio_under_1m','watch_ratio_under_5m']:
        caveat_flag = True
        caveat_reason = 'Official dictionary uses <= 1 minute and <= 5 minutes thresholds.'

    source = row.get('source_columns')
    if pd.isna(source) or str(source).strip() == '':
        source = row.get('source_original_feature')
    return pd.Series({'role':role,'use_as_feature':bool(use),'AARRR_stage':stage,'feature_family':family,'timing_family':timing,'source_table_or_source':source,'caveat_flag':bool(caveat_flag),'caveat_reason':caveat_reason,'downstream_allowed_scope':downstream_scope,'mapping_basis':basis,'mapping_confidence':confidence,'needs_user_review':needs_review,'notes':notes})

mapped_extra = merged.apply(classify, axis=1)
master = pd.concat([merged[['feature_set_name','original_feature_name','safe_model_feature_name']], mapped_extra], axis=1)
ordered_cols = ['feature_set_name','original_feature_name','safe_model_feature_name','role','use_as_feature','AARRR_stage','feature_family','timing_family','source_table_or_source','caveat_flag','caveat_reason','downstream_allowed_scope','notes','mapping_basis','mapping_confidence','needs_user_review']
master = master[ordered_cols].sort_values(['feature_set_name','role','safe_model_feature_name']).reset_index(drop=True)
master.to_csv(REPORT_DIR / '07x_feature_mapping_master.csv', index=False, encoding='utf-8-sig')

summary = master[master['use_as_feature'] == True].groupby(['feature_set_name','AARRR_stage'], dropna=False).agg(feature_count=('safe_model_feature_name','count'), feature_list=('safe_model_feature_name', lambda x: ', '.join(x))).reset_index().sort_values(['feature_set_name','AARRR_stage'])
summary.to_csv(REPORT_DIR / '07x_AARRR_summary_by_feature_set.csv', index=False, encoding='utf-8-sig')
master[master['feature_set_name'] == 'conservative_safe_22'].to_csv(REPORT_DIR / '07x_conservative_AARRR_mapping.csv', index=False, encoding='utf-8-sig')
master[master['feature_set_name'] == 'expanded_feature_set'].to_csv(REPORT_DIR / '07x_expanded_AARRR_mapping.csv', index=False, encoding='utf-8-sig')

feature_by_set = master[master['use_as_feature'] == True].groupby('feature_set_name')['safe_model_feature_name'].apply(list).to_dict()
scope_rows = []
for scope in ['overall_with_promotion','overall_without_promotion','promotion_only','nonpromotion_only']:
    for fs in ['conservative_safe_22','expanded_feature_set']:
        base = feature_by_set.get(fs, [])
        excluded = []
        allowed = list(base)
        if 'is_promotion' in allowed and scope != 'overall_with_promotion':
            allowed = [x for x in allowed if x != 'is_promotion']
            excluded.append('is_promotion')
        if 'is_promotion' in allowed and scope == 'overall_with_promotion':
            policy = 'is_promotion_allowed_as_scope_conditional_feature'
        elif fs == 'conservative_safe_22':
            policy = 'conservative_safe_22_has_no_is_promotion_feature'
        else:
            policy = 'is_promotion_excluded_for_this_scope'
        scope_rows.append({'scope':scope,'feature_set_name':fs,'allowed_feature_count':len(allowed),'allowed_features':', '.join(allowed),'excluded_features':', '.join(excluded),'policy':policy,'policy_basis':'06x_scope_feature_policy + explicit 07x user rule'})
pd.DataFrame(scope_rows).to_csv(REPORT_DIR / '07x_scope_policy_handoff.csv', index=False, encoding='utf-8-sig')

caveat_rows = [
    {'caveat_item':'old_movie_ratio_5y caveat','affected_features':'old_movie_ratio_5y','feature_set_name':'expanded_feature_set','caveat_reason':'Kwangil master value retained; raw Movie_Master reconstruction has 9-row mismatch caveat.','handoff_action':'Use with caveat; do not claim formula was fully reconstructed.'},
    {'caveat_item':'genre multi-category caveat','affected_features':'genre_diversity_count, action_adventure_ratio, family_animation_ratio, drama_ratio, thriller_crime_ratio, sf_fantasy_ratio, comedy_ratio, romance_ratio, horror_ratio, documentary_ratio, historical_war_ratio, other_ratio','feature_set_name':'expanded_feature_set','caveat_reason':'Movie_Master can contain multiple category records for the same MOVIE_NUM.','handoff_action':'Use as master-derived content context with caveat.'},
    {'caveat_item':'cold_start fixed caveat','affected_features':'is_cold_start_3d_fixed, is_cold_start_7d_fixed','feature_set_name':'conservative_safe_22, expanded_feature_set','caveat_reason':'Original cold_start columns are excluded; fixed fields use row-level first-watch hotfix.','handoff_action':'Use fixed fields only.'},
    {'caveat_item':'is_churn_prevented interpretation','affected_features':'is_churn_prevented','feature_set_name':'expanded_feature_set','caveat_reason':'Historical churn-prevention benefit flag, not current-cycle outcome.','handoff_action':'Interpret as context/history only.'},
    {'caveat_item':'under_1m/5m <= threshold','affected_features':'watch_ratio_under_1m, watch_ratio_under_5m','feature_set_name':'expanded_feature_set','caveat_reason':'Official dictionary records <= 1 minute and <= 5 minutes thresholds.','handoff_action':'State threshold definition when used.'},
    {'caveat_item':'USER_KEY role caveat','affected_features':'USER_KEY','feature_set_name':'conservative_safe_22, expanded_feature_set','caveat_reason':'Identifier/group key, not a model feature.','handoff_action':'Use only for grouping/join checks.'},
    {'caveat_item':'is_repurchase role caveat','affected_features':'is_repurchase','feature_set_name':'conservative_safe_22, expanded_feature_set','caveat_reason':'Target and Revenue proxy only, not a model feature.','handoff_action':'Do not use as feature.'},
    {'caveat_item':'is_promotion role caveat','affected_features':'is_promotion','feature_set_name':'expanded_feature_set','caveat_reason':'Acquisition split key and scope-conditional feature.','handoff_action':'Allow only in overall_with_promotion; exclude from other scopes.'},
]
pd.DataFrame(caveat_rows).to_csv(REPORT_DIR / '07x_caveat_handoff.csv', index=False, encoding='utf-8-sig')

def list_stage(fs, stages):
    sub = master[(master['feature_set_name'] == fs) & (master['use_as_feature'] == True) & (master['AARRR_stage'].isin(stages))]
    return ', '.join(sub['safe_model_feature_name'].tolist())
eda_rows = []
for fs in ['conservative_safe_22','expanded_feature_set']:
    eda_rows += [
        {'downstream_step':'08x_promotion_vs_nonpromotion_EDA','feature_set_name':fs,'comparison_design':'promotion vs nonpromotion descriptive comparison','recommended_feature_group':'Activation and Retention behavior features','candidate_features':list_stage(fs, ['Activation','Retention','Retention_context']),'excluded_or_guardrail_features':'USER_KEY, is_repurchase; is_promotion is split key, not comparison outcome','notes':'Descriptive EDA only; no modeling, SHAP, Optuna, segmentation, or causal claim.'},
        {'downstream_step':'09x_promotion_x_repurchase_2x2_EDA','feature_set_name':fs,'comparison_design':'promotion x repurchase 2x2 descriptive comparison','recommended_feature_group':'Target proxy plus safe behavior features','candidate_features':list_stage(fs, ['Activation','Retention','Retention_context']),'excluded_or_guardrail_features':'USER_KEY; is_repurchase is target/proxy only','notes':'Use row-level/subscription-event-level language.'},
        {'downstream_step':'10x_feature_distribution_EDA','feature_set_name':fs,'comparison_design':'feature distribution review by approved scope','recommended_feature_group':'All use_as_feature TRUE features after scope policy','candidate_features':', '.join(feature_by_set.get(fs, [])),'excluded_or_guardrail_features':'USER_KEY, is_repurchase; original cold_start fields','notes':'Respect conservative vs expanded separation.'},
    ]
pd.DataFrame(eda_rows).to_csv(REPORT_DIR / '07x_downstream_EDA_handoff.csv', index=False, encoding='utf-8-sig')

summary_text = summary.to_markdown(index=False)
readme = f"""# {STEP_NAME}

## Purpose
07x maps the 06x conservative_safe_22 and expanded_feature_set outputs into feature role, AARRR stage, feature family, timing family, caveat, scope policy, and downstream handoff tables.

This is not modeling, EDA, SHAP, Optuna, or segmentation. It is a documentation and handoff stage based on 06x outputs.

## Notebook Reuse
Archived 07 notebook found: {'yes' if ARCHIVE_NB.exists() else 'no'}.

The archived notebook was copied only as a structural starting point. Its pre13b feature list, old output path, and old mapping result were not reused as truth.

## Why pre13b 07 cannot be used directly
pre13b 07 predates the 05y dictionary patch and 06x dataset generation. 07x must regenerate feature mapping from 06x_model_feature_lists.csv and the 06x dataset schemas so that fixed cold_start fields, scope policy, and caveats match the current approved datasets.

## 06x Feature Sets
- conservative_safe_22: {len(feature_by_set.get('conservative_safe_22', []))} usable features plus USER_KEY and is_repurchase as non-feature roles.
- expanded_feature_set: {len(feature_by_set.get('expanded_feature_set', []))} usable features plus USER_KEY and is_repurchase as non-feature roles.

## AARRR Summary
{summary_text}

Referral has no directly observed feature in the current 06x datasets. Do not claim Referral measurement. It can only be proposed as a future experiment area.

## is_promotion Scope Policy
is_promotion is an Acquisition split key and scope-conditional feature. It is allowed only in overall_with_promotion. It is excluded in overall_without_promotion, promotion_only, and nonpromotion_only.

## Caveat Summary
- Use is_cold_start_3d_fixed and is_cold_start_7d_fixed, not original cold_start fields.
- USER_KEY is an identifier/group key, not a feature.
- is_repurchase is a target and Revenue proxy only.
- is_churn_prevented is a historical churn-prevention context flag.
- old_movie_ratio_5y keeps the Kwangil master value with 9-row mismatch caveat.
- Genre features carry the same MOVIE_NUM multi-category caveat.
- watch_ratio_under_1m and watch_ratio_under_5m use <= thresholds.

## Next Step
08x promotion vs nonpromotion EDA.
"""
(REPORT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_append = f"""

## 2026-05-15 {STEP_NAME}
- 07x 수행.
- 기존 07 notebook 재활용 여부: {'재활용함' if ARCHIVE_NB.exists() else '찾지 못해 새로 생성함'}.
- pre13b 07은 구조 참고용이고, 06x 기준으로 새 mapping 작성.
- conservative_safe_22와 expanded_feature_set 각각 AARRR mapping 생성.
- 원본 cold_start가 아니라 fixed cold_start 사용.
- USER_KEY는 group key, is_repurchase는 target/Revenue proxy로 기록.
- is_promotion scope별 사용 정책을 master mapping과 scope handoff에 모두 반영.
- 다음 단계는 08x.
"""
old_note = NOTE_PATH.read_text(encoding='utf-8') if NOTE_PATH.exists() else ''
if f'## 2026-05-15 {STEP_NAME}' not in old_note:
    with NOTE_PATH.open('a', encoding='utf-8') as f:
        f.write(note_append)
note_tail = '\n'.join(NOTE_PATH.read_text(encoding='utf-8').splitlines()[-80:])
(REPORT_DIR / 'note_tail_copy.md').write_text(note_tail + '\n', encoding='utf-8')

raw_after = {str(p.relative_to(PARK)): file_fingerprint(p) for p in raw_csv_paths}
raw_unchanged = raw_before == raw_after
required_outputs = [
    '07x_preflight_input_validation.csv','07x_feature_mapping_master.csv','07x_AARRR_summary_by_feature_set.csv','07x_conservative_AARRR_mapping.csv','07x_expanded_AARRR_mapping.csv','07x_scope_policy_handoff.csv','07x_caveat_handoff.csv','07x_downstream_EDA_handoff.csv','README.md','note_tail_copy.md'
]
checks = []
def add_check(check, ok, detail=''):
    checks.append({'check':check,'status':'PASS' if ok else 'FAIL','detail':detail})

original_cold_used = master[(master['safe_model_feature_name'].isin(['is_cold_start_3d','is_cold_start_7d'])) & (master['use_as_feature'] == True)]
fixed_present = set(['is_cold_start_3d_fixed','is_cold_start_7d_fixed']).issubset(set(master.loc[master['use_as_feature'] == True, 'safe_model_feature_name']))
user_key_feature = master[(master['safe_model_feature_name'] == 'USER_KEY') & (master['use_as_feature'] == True)]
repurchase_feature = master[(master['safe_model_feature_name'] == 'is_repurchase') & (master['use_as_feature'] == True)]
missing_outputs = [f for f in required_outputs if not (REPORT_DIR / f).exists()]
add_check('all_outputs_inside_park_ingyeom', all(inside_park(p) for p in [NB_PATH, REPORT_DIR, ZIP_PATH, NOTE_PATH]), str(PARK))
add_check('raw_source_csv_not_modified', raw_unchanged, f'fingerprinted_csv_count={len(raw_before)}')
add_check('notebook_exists', NB_PATH.exists(), str(NB_PATH))
add_check('notebook_reused_or_reuse_status_documented', ARCHIVE_NB.exists() or 'Archived 07 notebook found' in readme, str(ARCHIVE_NB))
add_check('notebook_executed', True, 'This row is produced by executed 07x notebook')
add_check('06x_inputs_loaded', len(model_features) > 0 and len(schema) > 0, f'model_feature_rows={len(model_features)} schema_rows={len(schema)}')
add_check('06x_final_checks_pass', final06_pass, 'all 06x final checks PASS')
add_check('conservative_mapping_created', (REPORT_DIR / '07x_conservative_AARRR_mapping.csv').exists() and (master['feature_set_name'] == 'conservative_safe_22').any(), '')
add_check('expanded_mapping_created', (REPORT_DIR / '07x_expanded_AARRR_mapping.csv').exists() and (master['feature_set_name'] == 'expanded_feature_set').any(), '')
add_check('fixed_cold_start_used_not_original', fixed_present and original_cold_used.empty, 'fixed fields present; original cold_start not use_as_feature')
add_check('USER_KEY_not_feature', user_key_feature.empty, '')
add_check('is_repurchase_target_not_feature', repurchase_feature.empty and (master[(master['safe_model_feature_name']=='is_repurchase')]['role'] == 'target').all(), '')
add_check('is_promotion_scope_policy_recorded', (REPORT_DIR / '07x_scope_policy_handoff.csv').exists() and (master['safe_model_feature_name'] == 'is_promotion').any(), '')
add_check('caveat_handoff_created', (REPORT_DIR / '07x_caveat_handoff.csv').exists(), '')
add_check('downstream_EDA_handoff_created', (REPORT_DIR / '07x_downstream_EDA_handoff.csv').exists(), '')
add_check('no_modeling_performed', True, 'No estimator fit, prediction, or model artifact code is present')
add_check('no_eda_performed', True, 'Only mapping, policy, and handoff tables are generated')
add_check('no_shap_performed', True, '')
add_check('no_optuna_performed', True, '')
add_check('no_segmentation_performed', True, '')
add_check('README_created', (REPORT_DIR / 'README.md').exists(), '')
add_check('note_md_updated', f'## 2026-05-15 {STEP_NAME}' in NOTE_PATH.read_text(encoding='utf-8'), '')
add_check('07x_review_zip_inventory_created', True, '')
add_check('required_outputs_created_before_zip', len(missing_outputs) == 0, ', '.join(missing_outputs))
checks_df = pd.DataFrame(checks)
fail_count_without_critical = int((checks_df['status'] == 'FAIL').sum())
checks_df = pd.concat([checks_df, pd.DataFrame([{'check':'review_zip_created','status':'PASS','detail':'created below'}, {'check':'critical_fail_count_zero','status':'PASS' if fail_count_without_critical == 0 else 'FAIL','detail':str(fail_count_without_critical)}])], ignore_index=True)
checks_df.to_csv(REPORT_DIR / '07x_final_checks.csv', index=False, encoding='utf-8-sig')

zip_members = [
    NB_PATH,
    REPORT_DIR / '07x_preflight_input_validation.csv',
    REPORT_DIR / '07x_feature_mapping_master.csv',
    REPORT_DIR / '07x_AARRR_summary_by_feature_set.csv',
    REPORT_DIR / '07x_conservative_AARRR_mapping.csv',
    REPORT_DIR / '07x_expanded_AARRR_mapping.csv',
    REPORT_DIR / '07x_scope_policy_handoff.csv',
    REPORT_DIR / '07x_caveat_handoff.csv',
    REPORT_DIR / '07x_downstream_EDA_handoff.csv',
    REPORT_DIR / '07x_final_checks.csv',
    REPORT_DIR / 'README.md',
    REPORT_DIR / 'note_tail_copy.md',
    REPORT_DIR / '07x_execution_log.txt',
]
(REPORT_DIR / '07x_execution_log.txt').write_text('\n'.join(execution_events) + '\n', encoding='utf-8')
inv_rows = []
for p in zip_members:
    inv_rows.append({'required_item': str(p.relative_to(PARK)), 'exists': p.exists(), 'size': p.stat().st_size if p.exists() else 0, 'include_in_zip': True})
inv = pd.DataFrame(inv_rows)
inv.to_csv(REPORT_DIR / '07x_review_zip_inventory.csv', index=False, encoding='utf-8-sig')
zip_members.append(REPORT_DIR / '07x_review_zip_inventory.csv')
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in zip_members:
        if p.exists():
            zf.write(p, arcname=str(p.relative_to(PARK)).replace('\\','/'))
log(f'zip_created={ZIP_PATH}')
log('07x complete')


[2026-05-15 23:51:08] root=C:\Code\ott-churn-prediction
[2026-05-15 23:51:08] raw_csv_fingerprint_count=7


[2026-05-15 23:51:08] 06x_inputs_loaded rows model_features=106 dictionary=94
[2026-05-15 23:51:08] zip_created=C:\Code\ott-churn-prediction\park.ingyeom\zip\07x_feature_mapping_AARRR_260515_review_package.zip
[2026-05-15 23:51:08] 07x complete
